# Results for model: gotocompany/gemma-2-9b-cpt-sahabatai-instruct

In [ ]:
import polars as pl
import xgboost as xgb

df = pl.read_parquet('./jane_street_data/train.parquet').reset_index()

top_quantile_rolling = df.groupby('date_id', ['feature_00', 'responder_6']) \
    .agg({'feature_00': 'top_quantile', 'responder_6': 'mean'}) \
    .compute()
    
top_quantile_rolling = top_quantile_rolling.rename(columns={'feature_00': 'global_top_quantile'})
top_quantile_rolling['global_top_quantile'] = top_quantile_rolling['global_top_quantile'] / top_quantile_rolling['responder_6']

df = df.merge(top_quantile_rolling, on=['date_id'], how='inner')

X = df[['date_id', 'feature_00']].to_frame()
y = df[['responder_6']]

model = xgb.XGBRegressor()
model.fit(X, y)